In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "pdf.fonttype": 42, "svg.fonttype": "none", "figure.dpi": 110})

ROI = "HIP"
CHEMS = None
SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

df = pd.read_parquet(f"{SLIM_DIR}/{ROI}_slim.parquet")
if CHEMS:
    df = df[df["chem"].isin(CHEMS)]
genes = [g for g in GRMS if g in df.columns]
cls_num = pd.to_numeric(df["class"].astype(str).str[:2], errors="coerce").fillna(0)
df["neuron"] = ~cls_num.between(30, 34)
os.makedirs(FIG_DIR, exist_ok=True)
tag = ROI + ("_" + "-".join(c.replace("WMB-10X", "") for c in CHEMS) if CHEMS else "")
print(ROI, len(df), "cells;", df["subclass"].nunique(), "subclasses; chem:", df["chem"].value_counts().to_dict())

d = df.dropna(subset=["allen_x", "allen_y"])
if len(d) > 150000:
    d = d.sample(150000, random_state=0)
ncol = 4
nrow = int(np.ceil(len(genes) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.6 * nrow))
for ax, g in zip(np.atleast_1d(axes).ravel(), genes):
    o = np.argsort(d[g].to_numpy())
    vmax = max(np.percentile(d[g], 99.5), 0.1)
    sc = ax.scatter(d["allen_x"].to_numpy()[o], d["allen_y"].to_numpy()[o], c=d[g].to_numpy()[o], s=1, cmap="viridis", vmin=0, vmax=vmax, linewidths=0, rasterized=True)
    ax.set_title(g, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    for s_ in ax.spines.values():
        s_.set_visible(False)
    cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.01)
    cb.ax.tick_params(labelsize=7)
fig.suptitle(f"{ROI}: log2(CPM+1) on Allen UMAP", fontsize=11)
fig.savefig(f"{FIG_DIR}/{tag}_fig3_umap_grm.png", dpi=300, bbox_inches="tight")
plt.show()